# Next Priority Runs — Exp43 / VS r32

Outputs are Drive-only. This notebook does not update `PROGRESS.md` and does not push result CSVs to GitHub.

Preset order:
- `p0_exp43_s24`: Exp43 C0/C1 on S24, VS init r=32
- `p1_exp43_s01`: Exp43 C0/C1 on S01, VS init r=32
- `p2_vs_s28`: S28 r=32 SD LoRA VS generation
- `p3_vs_s02`: S02 r=32 SD LoRA VS generation


In [1]:
# [1] GPU check
import torch
assert torch.cuda.is_available(), 'GPU 없음: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)
print('GPU  ', torch.cuda.get_device_name(0))


torch 2.11.0+cu128
GPU   NVIDIA L4


In [2]:
# [2] Mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if os.path.exists(WORKDIR) and os.path.exists(os.path.join(WORKDIR, '.git')):
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['rm', '-rf', WORKDIR], check=True)
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-5'], check=False)


Mounted at /content/drive


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
# [3] Select preset and launch background run
# Change PRESET one at a time. Do not run multiple presets simultaneously on one GPU.
PRESET = 'p1_exp43_s01' # 'p0_exp43_s24'  # p0_exp43_s24 | p1_exp43_s01 | p2_vs_s28 | p3_vs_s02

# Exp43 default uses VI/class=60 => 432 train trials. Set FULL_VI=True for all VI trials.
FULL_VI = False

cmd = f"python -u launch_next_priority_runs.py --preset {PRESET}"
if FULL_VI:
    cmd += ' --full_vi'

print(cmd)
!cd /content/vsvi_project && {cmd}


In [ ]:
# [4] Monitor active process and GPU
!pgrep -af 'train_exp43_vi_lora.py|train_vs_re_lora_gen.py' || true
!nvidia-smi
!ls -lt /content/drive/MyDrive/vsvi_data/logs | head -20


In [ ]:
# [5] Tail latest log for the selected preset
import glob, os
patterns = {
    'p0_exp43_s24': '/content/drive/MyDrive/vsvi_data/logs/exp43_s24_c0c1_r32_tok16_*.log',
    'p1_exp43_s01': '/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_*.log',
    'p2_vs_s28': '/content/drive/MyDrive/vsvi_data/logs/vs_s28_lora_r32_tok16_*.log',
    'p3_vs_s02': '/content/drive/MyDrive/vsvi_data/logs/vs_s02_lora_r32_tok16_*.log',
}
logs = sorted(glob.glob(patterns[PRESET]), key=os.path.getmtime, reverse=True)
assert logs, f'No log found for {PRESET}'
LOG = logs[0]
print('LOG=', LOG)
!tail -n 120 "{LOG}"


In [ ]:
# [6] Find Drive-only result CSVs
!echo '=== Exp43 VI results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name 'results_exp43_vi_lora.csv' -ls 2>/dev/null | tail -30
!echo '=== VS results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen -name 'results_lora_gen.csv' -ls 2>/dev/null | tail -30


In [ ]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

OSError: [Errno 125] Operation canceled: '/content/drive/.Encrypted/.shortcut-targets-by-id'

In [ ]:
%%bash
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ Drive OK" || echo "❌ Drive 문제"

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz
✅ Drive OK


In [ ]:
%%bash
# 환경 복구
cd /content
git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project 2>/dev/null
cd /content/vsvi_project && git pull
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re
ln -sfn /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino /content/vsvi_project/checkpoints_vsre_dino
pip uninstall -y torchao 2>/dev/null

ls /content/vsvi_project/preproc_vi_re/preproc_subj_02_1.npz && echo "✅ Data OK" || exit 1

# S02 Exp43
python -u train_exp43_vi_lora.py \
  --subject_ids 2 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260707_130522_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 135 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s02_c0c1_r32_tok16_full_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project
rm -rf /content/vsvi_project/preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_$(date +%Y%m%d_%H%M%S).log

In [ ]:
import os, shutil
# 잔여 제거
try:
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
except:
    pass

from google.colab import drive
drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [ ]:
import os
# 마운트 상태 확인
print("MyDrive:", os.path.exists('/content/drive/MyDrive'))
print("내용:", os.listdir('/content/drive/MyDrive')[:10] if os.path.exists('/content/drive/MyDrive') else "없음")

MyDrive: True
내용: ['Quince · SlidesCarnival의 사본.gslides', 'Hyperscanning_', 'DSI 24.pptx', 'EEG&연구설명 발표자료.pptx', 'Print', 'vsvi_data', 'MI_loso_project', 'project.tar.gz']


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data'))

Mounted at /content/drive
vsvi_data: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re'))

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/vsvi_data'):
    from google.colab import drive
    drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))
print("S09 파일:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("S09 vs:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
S09 vs: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print("vi_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_09_1.npz'))
print("vs_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Mounted at /content/drive
vi_re: True
vs_re: True


In [ ]:
%%bash
cd /content/vsvi_project

# 실제 디렉터리 제거 (rm -rf)
rm -rf /content/vsvi_project/preproc_vs_re
rm -rf /content/vsvi_project/preproc_vi_re

# symlink 생성
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

# 확인
ls -la preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || echo "❌ 실패"

-rw------- 1 root root 16132799 Jul  9 18:05 preproc_vs_re/preproc_subj_09_1.npz
✅ OK


In [ ]:
%%bash
cd /content/vsvi_project 2>/dev/null || (cd /content && git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project && cd vsvi_project)
cd /content/vsvi_project
rm -f preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

ls preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || exit 1

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_$(date +%Y%m%d_%H%M%S).log

preproc_vs_re/preproc_subj_09_1.npz
✅ OK
[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vi

In [ ]:
%%bash
cd /content/vsvi_project

# S09 데이터에 NaN이나 이상값이 있는지 확인
python3 -c "
import numpy as np
import glob

for sid in [9, 28, 29]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vs_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} ({len(files)} sessions) ===')
    for f in files[:2]:
        d = np.load(f)
        keys = list(d.keys())
        X = d[keys[0]]
        print(f'  {f.split(\"/\")[-1]}: shape={X.shape}, NaN={np.isnan(X).any()}, min={X.min():.3f}, max={X.max():.3f}, mean={X.mean():.3f}')
"

=== S09 (4 sessions) ===
  preproc_subj_09_1.npz: shape=(135, 32, 2048), NaN=False, min=-242.250, max=397.500, mean=-0.609
  preproc_subj_09_2.npz: shape=(135, 32, 2048), NaN=False, min=-545.000, max=596.500, mean=-0.358
=== S28 (6 sessions) ===
  preproc_subj_28_1.npz: shape=(135, 32, 2048), NaN=False, min=-inf, max=inf, mean=nan
  preproc_subj_28_2.npz: shape=(135, 32, 2048), NaN=False, min=-158.625, max=285.750, mean=-0.360
=== S29 (5 sessions) ===
  preproc_subj_29_1.npz: shape=(135, 32, 2048), NaN=False, min=-219.750, max=360.000, mean=-0.312
  preproc_subj_29_2.npz: shape=(135, 32, 2048), NaN=False, min=-276.250, max=380.250, mean=-0.338


/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:127: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vs_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf 개수={n_inf}, shape={X.shape}')
"

preproc_subj_28_1.npz: inf 개수=44767, shape=(135, 32, 2048)
preproc_subj_28_2.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_3.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_4.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_5.npz: inf 개수=0, shape=(108, 32, 2048)
preproc_subj_28_6.npz: inf 개수=0, shape=(135, 32, 2048)


In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 09  (sessions=4)
  [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(108, 32, 2048)
  [dataset] S09  trials=5

In [ ]:
%%bash
# S28 sess1을 백업 (로더가 안 읽도록 확장자 변경)
mv /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_5.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_6.npz


In [ ]:
%%bash
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [28]
[INFO] Session counts: {28: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 28  (sessions=5)
  [dataset] S28 sess2: npz  shape=(135, 32, 2048)
  [dataset] S28 sess3: npz  shape=(135, 32, 2048)
  [dataset] S28 sess4: npz  shape=(135, 32, 2048)
  [dataset] S28 sess5: npz  shape=(108, 32, 2048)
  [dataset] S28 sess6: 

In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [29]
[INFO] Session counts: {29: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 29  (sessions=5)
  [dataset] S29 sess1: npz  shape=(135, 32, 2048)
  [dataset] S29 sess2: npz  shape=(135, 32, 2048)
  [dataset] S29 sess3: npz  shape=(135, 32, 2048)
  [dataset] S29 sess4: npz  shape=(135, 32, 2048)
  [dataset] S29 sess5: 

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project

python -u train_vs_re_lora_gen.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s28_r32_$(date +%Y%m%d_%H%M%S).log

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migratin

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 90 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_

In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vi_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    n_nan = np.isnan(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf={n_inf}, nan={n_nan}, shape={X.shape}')
"

bash: line 1: cd: /content/vsvi_project: No such file or directory


In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
for sid in [28, 29, 9]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vi_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} VI ===')
    for f in files:
        d = np.load(f)
        X = d[list(d.keys())[0]]
        print(f'  {f.split(\"/\")[-1]}: inf={np.isinf(X).sum()}, nan={np.isnan(X).sum()}')
"

=== S28 VI ===
  preproc_subj_28_1.npz: inf=0, nan=0
  preproc_subj_28_2.npz: inf=0, nan=0
  preproc_subj_28_3.npz: inf=0, nan=0
  preproc_subj_28_4.npz: inf=0, nan=0
  preproc_subj_28_5.npz: inf=33673, nan=0
  preproc_subj_28_6.npz: inf=0, nan=0
=== S29 VI ===
  preproc_subj_29_1.npz: inf=0, nan=0
  preproc_subj_29_2.npz: inf=0, nan=0
  preproc_subj_29_3.npz: inf=0, nan=0
  preproc_subj_29_4.npz: inf=0, nan=0
  preproc_subj_29_5.npz: inf=0, nan=0
=== S09 VI ===
  preproc_subj_09_1.npz: inf=0, nan=0
  preproc_subj_09_2.npz: inf=0, nan=0
  preproc_subj_09_3.npz: inf=0, nan=0
  preproc_subj_09_4.npz: inf=0, nan=0


In [ ]:
%%bash
# S28 VI sess5 제외
mv /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_1.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_6.npz


mv: cannot stat '/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz': No such file or directory


In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
[INFO] local DINO cache not found, downloading facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 228M

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s29_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
[INFO]  local cache not found, downloading from facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 29 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s29_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

Process is terminated.


In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_185832_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s09_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/

In [ ]:
%%bash
sed -n '164,185p' /content/vsvi_project/train_vs_re_lora_gen.py
echo "=== make_schedule 반환 ==="
grep -n "def make_schedule" /content/vsvi_project/train_vs_re_latent_gen.py
sed -n '/def make_schedule/,/return/p' /content/vsvi_project/train_vs_re_latent_gen.py | head -25

def sample_sd_ddim(unet, cond_proj, eeg_lat, acp, T, steps=50, device='cuda', guidance=1.0):
    """DDIM sampling with SD 1.5 UNet conditioned on EEG."""
    B = eeg_lat.size(0)
    cond_tokens = cond_proj(eeg_lat)  # (B, N_tok, 768)

    seq = list(range(0, T, T // steps))
    x = torch.randn(B, 4, 64, 64, device=device)

    for i in reversed(range(len(seq))):
        t_val = seq[i]
        t_tensor = torch.full((B,), t_val, dtype=torch.long, device=device)
        noise_pred = unet(x, t_tensor, encoder_hidden_states=cond_tokens).sample
        a  = acp[seq[i]]
        a_ = acp[seq[i-1]] if i > 0 else torch.tensor(1.0, device=device)
        x0_pred = (x - (1-a).sqrt() * noise_pred) / a.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)
        x = a_.sqrt() * x0_pred + (1 - a_).sqrt() * noise_pred
    return x


@torch.no_grad()
def evaluate_lora(unet, cond_proj, eeg_enc, test_loader, vae, dino,
=== make_schedule 반환 ===
267:def make_schedule(T, device):
def make_schedule(T, device):
    

In [ ]:
%%bash
grep -n "acp\|make_schedule" /content/vsvi_project/train_exp43_vi_lora.py | head -10

40:from train_vs_re_latent_gen import build_eeg_encoder, make_schedule
337:    acp,
423:            xt, noise = ddpm_q_sample(x0, t, acp)
481:        acp,
600:    _, _, acp = make_schedule(args.num_timesteps, device)
675:                acp,


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi")
manifests = sorted((root / "seed42" / "S09").rglob("generation_manifest.csv"))

print("S09 manifests:", len(manifests))
for p in manifests:
    print(p)

S09 manifests: 3
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/vi_only/generation_manifest.csv


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi")

CKPTS = {
    1: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S01/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt",
    },
    2: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S02/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    28: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S28/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt",
    },
    29: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S29/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt",
    },
}

for sid, c in CKPTS.items():
    done = sorted((OUT_ROOT / "seed42" / f"S{sid:02d}").rglob("generation_manifest.csv"))
    if len(done) >= 3:
        print(f"[SKIP] S{sid:02d}: manifests={len(done)}")
        continue

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "2",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0

[SKIP] S01: manifests=3
[SKIP] S02: manifests=3
[SKIP] S18: manifests=3
[SKIP] S28: manifests=3
[SKIP] S29: manifests=3


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOTS = [
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi"),
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24"),
]

manifests = []
for root in ROOTS:
    manifests.extend(sorted(root.rglob("generation_manifest.csv")))

print("manifests:", len(manifests))
for p in manifests:
    print(p)

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen = pd.DataFrame(rows)
display(gen.sort_values(["subject", "encoder", "conditioning"]))

manifests: 21
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/vi_only/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/vi_only/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_d

,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S01,full_vs_to_vi,correct,18,0.111111,0.111111,0.388889,"[0, 0, 1, 2, 2, 3, 0, 3, 7]"
1,S01,full_vs_to_vi,shuffled,18,0.055556,0.222222,0.500000,"[0, 1, 2, 1, 3, 0, 1, 1, 9]"
2,S01,full_vs_to_vi,zero,18,0.111111,NaN,0.388889,"[0, 1, 2, 3, 7, 2, 0, 1, 2]"
3,S01,gated_residual,correct,18,0.166667,0.166667,0.388889,"[0, 0, 1, 1, 3, 4, 2, 0, 7]"
4,S01,gated_residual,shuffled,18,0.111111,0.166667,0.444444,"[0, 0, 1, 1, 1, 4, 2, 1, 8]"
...,...,...,...,...,...,...,...,...
49,S29,gated_residual,shuffled,18,0.111111,0.166667,0.222222,"[3, 0, 3, 4, 1, 3, 1, 1, 2]"
50,S29,gated_residual,zero,18,0.166667,NaN,0.333333,"[0, 0, 2, 1, 5, 2, 0, 2, 6]"
51,S29,vi_only,correct,18,0.277778,0.277778,0.277778,"[3, 0, 1, 2, 1, 5, 0, 1, 5]"
52,S29,vi_only,shuffled,18,0.111111,0.166667,0.277778,"[3, 2, 2, 2, 0, 5, 1, 1, 2]"


In [ ]:
pivot = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot["shuffled_minus_correct"] = pivot["shuffled"] - pivot["correct"]

display(pivot.reset_index().sort_values(["encoder", "subject"]))

group = (
    pivot.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(group)

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.222222,0.111111
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.277778,0.222222
15,S28,full_vs_to_vi,0.111111,0.166667,0.055556
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.166667,0.000000
4,S02,gated_residual,0.166667,0.111111,-0.055556
7,S09,gated_residual,0.055556,0.166667,0.111111


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,0.055556,0.055556,0.101430
gated_residual,7,0.007937,0.000000,0.059391
vi_only,7,-0.015873,0.000000,0.052844


In [ ]:
orig = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig["shuffled_minus_correct"] = orig["shuffled"] - orig["correct"]

display(orig.reset_index().sort_values(["encoder", "subject"]))

orig_group = (
    orig.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group)

print(orig_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.055556,-0.055556
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.055556,0.000000
15,S28,full_vs_to_vi,0.111111,0.111111,0.000000
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.111111,-0.055556
4,S02,gated_residual,0.166667,0.166667,0.000000
7,S09,gated_residual,0.055556,0.055556,0.000000


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,-0.007937,0.000000,0.059391
gated_residual,7,-0.015873,0.000000,0.069643
vi_only,7,-0.023810,-0.055556,0.089908


                count      mean    median       std
encoder                                            
full_vs_to_vi       7 -0.007937  0.000000  0.059391
gated_residual      7 -0.015873  0.000000  0.069643
vi_only             7 -0.023810 -0.055556  0.089908


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

SUBJECTS = [9, 18, 24]

CKPTS_SPC5 = {
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    24: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
    },
}

for sid in SUBJECTS:
    c = CKPTS_SPC5[sid]

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} samples_per_class=5 lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "5",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
        "--overwrite",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0


START S09 samples_per_class=5 lora_r=32
RETURN CODE: 1


AssertionError: 

In [ ]:
import glob
from pathlib import Path

patterns = [
    "/content/drive/MyDrive/vsvi_data/**/S09/**/vi_only/**/encoder_best.pt",
    "/content/drive/MyDrive/vsvi_data/**/S09/**/raw_tf/**/encoder_best.pt",
    "/content/drive/MyDrive/vsvi_data/**/S09/**/encoder_best.pt",
]

hits = []
for pat in patterns:
    hits.extend(glob.glob(pat, recursive=True))

hits = sorted(set(hits))

print("hits:", len(hits))
for p in hits:
    print(p)

hits: 12
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vs_pretrain/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S09/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S09/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vs_pretrain/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_pretrain/encoder_best.pt
/content/drive/MyDriv

In [ ]:
from pathlib import Path

CKPTS = {
    1: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S01/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt",
    },
    2: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S02/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt",
    },
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    28: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S28/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt",
    },
    29: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S29/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt",
    },
}

In [ ]:
CKPTS[24] = {
    "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
    "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
    "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
    "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
}

In [ ]:
from pathlib import Path

roots = [
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi"),
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24"),
    Path("/content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only"),
]

for root in roots:
    manifests = sorted(root.rglob("generation_manifest.csv")) if root.exists() else []
    print("\n", root)
    print("manifests:", len(manifests))
    for p in manifests:
        print(" ", p)


 /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi
manifests: 0

 /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24
manifests: 0

 /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only
manifests: 21
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S02/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S02

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
assert len(manifests) == 21

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen = pd.DataFrame(rows)
display(gen.sort_values(["subject", "encoder", "conditioning"]))

manifests: 21


,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S01,full_vs_to_vi,correct,18,0.111111,0.111111,0.388889,"[0, 0, 1, 2, 2, 3, 0, 3, 7]"
1,S01,full_vs_to_vi,shuffled,18,0.055556,0.222222,0.500000,"[0, 1, 2, 1, 3, 0, 1, 1, 9]"
2,S01,full_vs_to_vi,zero,18,0.111111,NaN,0.388889,"[0, 1, 2, 3, 7, 2, 0, 1, 2]"
3,S01,gated_residual,correct,18,0.166667,0.166667,0.388889,"[0, 0, 1, 1, 3, 4, 2, 0, 7]"
4,S01,gated_residual,shuffled,18,0.111111,0.166667,0.444444,"[0, 0, 1, 1, 1, 4, 2, 1, 8]"
...,...,...,...,...,...,...,...,...
49,S29,gated_residual,shuffled,18,0.111111,0.166667,0.222222,"[3, 0, 3, 4, 1, 3, 1, 1, 2]"
50,S29,gated_residual,zero,18,0.166667,NaN,0.333333,"[0, 0, 2, 1, 5, 2, 0, 2, 6]"
51,S29,vi_only,correct,18,0.277778,0.277778,0.277778,"[3, 0, 1, 2, 1, 5, 0, 1, 5]"
52,S29,vi_only,shuffled,18,0.111111,0.166667,0.277778,"[3, 2, 2, 2, 0, 5, 1, 1, 2]"


In [ ]:
pivot = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot["shuffled_minus_correct"] = pivot["shuffled"] - pivot["correct"]

display(pivot.reset_index().sort_values(["encoder", "subject"]))

source_group = (
    pivot.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group)
print(source_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.222222,0.111111
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.277778,0.222222
15,S28,full_vs_to_vi,0.111111,0.166667,0.055556
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.166667,0.000000
4,S02,gated_residual,0.166667,0.111111,-0.055556
7,S09,gated_residual,0.055556,0.166667,0.111111


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,0.055556,0.055556,0.101430
gated_residual,7,0.007937,0.000000,0.059391
vi_only,7,-0.015873,0.000000,0.052844


                count      mean    median       std
encoder                                            
full_vs_to_vi       7  0.055556  0.055556  0.101430
gated_residual      7  0.007937  0.000000  0.059391
vi_only             7 -0.015873  0.000000  0.052844


In [ ]:
orig = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig["shuffled_minus_correct"] = orig["shuffled"] - orig["correct"]

display(orig.reset_index().sort_values(["encoder", "subject"]))

orig_group = (
    orig.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group)
print(orig_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.055556,-0.055556
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.055556,0.000000
15,S28,full_vs_to_vi,0.111111,0.111111,0.000000
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.111111,-0.055556
4,S02,gated_residual,0.166667,0.166667,0.000000
7,S09,gated_residual,0.055556,0.055556,0.000000


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,-0.007937,0.000000,0.059391
gated_residual,7,-0.015873,0.000000,0.069643
vi_only,7,-0.023810,-0.055556,0.089908


                count      mean    median       std
encoder                                            
full_vs_to_vi       7 -0.007937  0.000000  0.059391
gated_residual      7 -0.015873  0.000000  0.069643
vi_only             7 -0.023810 -0.055556  0.089908


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

SUBJECTS = [9, 18, 24]

CKPTS_SPC5 = {
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    24: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
    },
}

for sid in SUBJECTS:
    c = CKPTS_SPC5[sid]

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} samples_per_class=5 lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "5",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
        "--overwrite",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0

In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

sid = 9
c = CKPTS_SPC5[sid]

m = re.search(r"lora_r(\d+)", c["generator"])
lora_r = int(m.group(1)) if m else 32

cmd = [
    sys.executable, "-u", SCRIPT,
    "--subject_id", str(sid),
    "--data_root", "/content/drive/MyDrive/vsvi_data/preproc_vi_re",
    "--img_root", "/content/vsvi_project/preproc_data_vi/images",
    "--vi_only_ckpt", c["vi_only"],
    "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
    "--gated_ckpt", c["gated"],
    "--generator_ckpt", c["generator"],
    "--out_root", "/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5",
    "--seed", "42",
    "--samples_per_class", "5",
    "--generations_per_trial", "1",
    "--ddim_steps", "30",
    "--lora_r", str(lora_r),
    "--lora_alpha", str(lora_r),
    "--fp16",
    "--overwrite",
]

result = subprocess.run(cmd, text=True, capture_output=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT LAST 5000\n", result.stdout[-5000:])
print("\nSTDERR LAST 5000\n", result.stderr[-5000:])

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
for p in manifests:
    print(p)

assert len(manifests) == 9

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'torchao'], returncode=0)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen5 = pd.DataFrame(rows)
display(gen5.sort_values(["subject", "encoder", "conditioning"]))

In [ ]:
pivot5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot5["shuffled_minus_correct"] = pivot5["shuffled"] - pivot5["correct"]

display(pivot5.reset_index().sort_values(["encoder", "subject"]))

source_group5 = (
    pivot5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group5)
print(source_group5.to_string())

In [ ]:
orig5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig5["shuffled_minus_correct"] = orig5["shuffled"] - orig5["correct"]

display(orig5.reset_index().sort_values(["encoder", "subject"]))

orig_group5 = (
    orig5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group5)
print(orig_group5.to_string())

In [3]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

print("exists:", ROOT.exists())
print("path:", ROOT)

exists: True
path: /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5


In [4]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

manifests = sorted(ROOT.rglob("generation_manifest.csv"))
metrics = sorted(ROOT.rglob("metrics.json"))
images = sorted(ROOT.rglob("*.png"))

print("manifests:", len(manifests))
for p in manifests:
    print(" ", p)

print("\nmetrics:", len(metrics))
for p in metrics:
    print(" ", p)

print("\nimages:", len(images))
print("first images:")
for p in images[:10]:
    print(" ", p)

manifests: 9
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S24/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S24/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/v

In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

rows = []

for sid in [9, 18, 24]:
    sdir = ROOT / "seed42" / f"S{sid:02d}"
    for enc in ["vi_only", "full_vs_to_vi", "gated_residual"]:
        manifest = sdir / enc / "generation_manifest.csv"
        metrics = sdir / enc / "metrics.json"
        img_dir = sdir / enc / "images"
        rows.append({
            "subject": f"S{sid:02d}",
            "encoder": enc,
            "manifest": manifest.exists(),
            "metrics": metrics.exists(),
            "n_images": len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0,
            "path": str(sdir / enc),
        })

check = pd.DataFrame(rows)
display(check)

,subject,encoder,manifest,metrics,n_images,path
0,S09,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
1,S09,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
2,S09,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
3,S18,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
4,S18,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
5,S18,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
6,S24,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
7,S24,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
8,S24,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
assert len(manifests) == 9

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen5 = pd.DataFrame(rows)
display(gen5.sort_values(["subject", "encoder", "conditioning"]))

manifests: 9


,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S09,full_vs_to_vi,correct,45,0.244444,0.244444,0.333333,"[2, 3, 15, 4, 8, 0, 0, 5, 8]"
1,S09,full_vs_to_vi,shuffled,45,0.066667,0.222222,0.422222,"[4, 4, 19, 1, 3, 3, 0, 6, 5]"
2,S09,full_vs_to_vi,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
3,S09,gated_residual,correct,45,0.066667,0.066667,0.511111,"[8, 3, 23, 1, 6, 0, 1, 1, 2]"
4,S09,gated_residual,shuffled,45,0.022222,0.088889,0.444444,"[9, 3, 20, 0, 2, 0, 0, 1, 10]"
5,S09,gated_residual,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
6,S09,vi_only,correct,45,0.088889,0.088889,0.555556,"[7, 2, 25, 0, 5, 1, 0, 0, 5]"
7,S09,vi_only,shuffled,45,0.022222,0.111111,0.533333,"[6, 1, 24, 2, 7, 0, 1, 0, 4]"
8,S09,vi_only,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
9,S18,full_vs_to_vi,correct,45,0.155556,0.155556,0.288889,"[3, 0, 3, 5, 5, 5, 2, 9, 13]"


In [7]:
pivot5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot5["shuffled_minus_correct"] = pivot5["shuffled"] - pivot5["correct"]

display(pivot5.reset_index().sort_values(["encoder", "subject"]))

source_group5 = (
    pivot5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group5)
print(source_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.244444,0.222222,-0.022222
3,S18,full_vs_to_vi,0.155556,0.111111,-0.044444
6,S24,full_vs_to_vi,0.088889,0.066667,-0.022222
1,S09,gated_residual,0.066667,0.088889,0.022222
4,S18,gated_residual,0.177778,0.133333,-0.044444
7,S24,gated_residual,0.044444,0.044444,0.000000
2,S09,vi_only,0.088889,0.111111,0.022222
5,S18,vi_only,0.088889,0.177778,0.088889
8,S24,vi_only,0.088889,0.066667,-0.022222


,count,mean,median,std
encoder,,,,
vi_only,3,0.029630,0.022222,0.055925
gated_residual,3,-0.007407,0.000000,0.033945
full_vs_to_vi,3,-0.029630,-0.022222,0.012830


                count      mean    median       std
encoder                                            
vi_only             3  0.029630  0.022222  0.055925
gated_residual      3 -0.007407  0.000000  0.033945
full_vs_to_vi       3 -0.029630 -0.022222  0.012830


In [8]:
orig5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig5["shuffled_minus_correct"] = orig5["shuffled"] - orig5["correct"]

display(orig5.reset_index().sort_values(["encoder", "subject"]))

orig_group5 = (
    orig5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group5)
print(orig_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.244444,0.066667,-0.177778
3,S18,full_vs_to_vi,0.155556,0.066667,-0.088889
6,S24,full_vs_to_vi,0.088889,0.177778,0.088889
1,S09,gated_residual,0.066667,0.022222,-0.044444
4,S18,gated_residual,0.177778,0.088889,-0.088889
7,S24,gated_residual,0.044444,0.177778,0.133333
2,S09,vi_only,0.088889,0.022222,-0.066667
5,S18,vi_only,0.088889,0.066667,-0.022222
8,S24,vi_only,0.088889,0.155556,0.066667


,count,mean,median,std
encoder,,,,
gated_residual,3,0.000000,-0.044444,0.117589
vi_only,3,-0.007407,-0.022222,0.067890
full_vs_to_vi,3,-0.059259,-0.088889,0.135780


                count      mean    median       std
encoder                                            
gated_residual      3  0.000000 -0.044444  0.117589
vi_only             3 -0.007407 -0.022222  0.067890
full_vs_to_vi       3 -0.059259 -0.088889  0.135780


In [9]:
dom5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="dominant_ratio",
)

dom5["shuffled_minus_correct"] = dom5["shuffled"] - dom5["correct"]

display(dom5.reset_index().sort_values(["encoder", "subject"]))

dom_group5 = (
    dom5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(dom_group5)
print(dom_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.333333,0.422222,0.088889
3,S18,full_vs_to_vi,0.288889,0.400000,0.111111
6,S24,full_vs_to_vi,0.444444,0.355556,-0.088889
1,S09,gated_residual,0.511111,0.444444,-0.066667
4,S18,gated_residual,0.288889,0.288889,0.000000
7,S24,gated_residual,0.488889,0.466667,-0.022222
2,S09,vi_only,0.555556,0.533333,-0.022222
5,S18,vi_only,0.444444,0.377778,-0.066667
8,S24,vi_only,0.466667,0.400000,-0.066667


,count,mean,median,std
encoder,,,,
full_vs_to_vi,3,0.037037,0.088889,0.109620
gated_residual,3,-0.029630,-0.022222,0.033945
vi_only,3,-0.051852,-0.066667,0.025660


                count      mean    median       std
encoder                                            
full_vs_to_vi       3  0.037037  0.088889  0.109620
gated_residual      3 -0.029630 -0.022222  0.033945
vi_only             3 -0.051852 -0.066667  0.025660


In [10]:
import pandas as pd
from pathlib import Path

rows = [
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "full_vs_to_vi",
        "source_following_delta": 0.055556,
        "original_label_delta": -0.007937,
        "dominant_delta": None,
        "interpretation": "weak positive source-following, unstable",
    },
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "gated_residual",
        "source_following_delta": 0.007937,
        "original_label_delta": -0.015873,
        "dominant_delta": None,
        "interpretation": "near zero source-following",
    },
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "vi_only",
        "source_following_delta": -0.015873,
        "original_label_delta": -0.023810,
        "dominant_delta": None,
        "interpretation": "no source-following",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "full_vs_to_vi",
        "source_following_delta": -0.029630,
        "original_label_delta": -0.059259,
        "dominant_delta": 0.037037,
        "interpretation": "conditioning changes output but not source-semantically",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "gated_residual",
        "source_following_delta": -0.007407,
        "original_label_delta": 0.000000,
        "dominant_delta": -0.029630,
        "interpretation": "weak generation conditioning",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "vi_only",
        "source_following_delta": 0.029630,
        "original_label_delta": -0.007407,
        "dominant_delta": -0.051852,
        "interpretation": "small inconsistent source-following",
    },
]

df = pd.DataFrame(rows)

OUT = Path("/content/drive/MyDrive/vsvi_data/downstream_generation_transfer_summary.csv")
df.to_csv(OUT, index=False)
display(df)
print("saved:", OUT)

,setting,n_subjects,encoder,source_following_delta,original_label_delta,dominant_delta,interpretation
0,spc2_all_subjects,7,full_vs_to_vi,0.055556,-0.007937,NaN,"weak positive source-following, unstable"
1,spc2_all_subjects,7,gated_residual,0.007937,-0.015873,NaN,near zero source-following
2,spc2_all_subjects,7,vi_only,-0.015873,-0.023810,NaN,no source-following
3,spc5_selected_subjects,3,full_vs_to_vi,-0.029630,-0.059259,0.037037,conditioning changes output but not source-sem...
4,spc5_selected_subjects,3,gated_residual,-0.007407,0.000000,-0.029630,weak generation conditioning
5,spc5_selected_subjects,3,vi_only,0.029630,-0.007407,-0.051852,small inconsistent source-following


saved: /content/drive/MyDrive/vsvi_data/downstream_generation_transfer_summary.csv


In [11]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data")

keywords = [
    "vi_rawtf_confirmatory_multi",
    "vi_rawtf_gated_residual_confirmatory",
    "vi_encoder_transfer_multi_exp26",
    "vi_tf_representation_ablation",
]

files = []

for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix in [".csv", ".json"]:
        s = str(p)
        if any(k in s for k in keywords):
            files.append(p)

print("files:", len(files))
for p in sorted(files):
    print(p)

files: 341
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vi_only/confusion.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vi_only/history.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vi_only/predictions.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vs_to_vi/confusion.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vs_to_vi/history.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vs_to_vi/metrics.json
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/vs_to_vi/predictions.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/zero_shot/confusion.csv
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S01/zero_shot/metrics.json
/con

In [12]:
import json
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/content/drive/MyDrive/vsvi_data")

SUBJECTS = [1, 2, 9, 18, 24, 28, 29]

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

rows = []

# JSON metrics 우선 수집
for p in ROOT.rglob("metrics.json"):
    s = str(p)

    if not any(k in s for k in [
        "vi_rawtf_confirmatory_multi",
        "vi_rawtf_gated_residual_confirmatory",
        "vi_tf_representation_ablation",
        "vi_encoder_transfer_multi_exp26",
    ]):
        continue

    subj_match = re.search(r"/S(\d{2})/", s)
    if not subj_match:
        continue

    sid = int(subj_match.group(1))
    if sid not in SUBJECTS:
        continue

    try:
        m = flatten(load_json(p))
    except Exception as e:
        print("failed:", p, e)
        continue

    if "/raw_tf/vi_only/" in s:
        method = "raw_tf_vi_only"
    elif "/raw_tf/vs_to_vi/" in s:
        method = "raw_tf_vs_to_vi"
    elif "/raw/vi_only/" in s:
        method = "raw_vi_only"
    elif "/raw/vs_to_vi/" in s:
        method = "raw_vs_to_vi"
    elif "gated_residual_vs_to_vi" in s:
        method = "gated_residual_vs_to_vi"
    elif "/vi_only/" in s:
        method = "vi_only_other"
    elif "/vs_to_vi/" in s:
        method = "vs_to_vi_other"
    else:
        method = "unknown"

    row = {
        "subject": sid,
        "method": method,
        "path": str(p),
    }

    for key in [
        "balanced_accuracy",
        "test_BAC",
        "VI_test_BAC",
        "target_VI_BAC",
        "top1",
        "VI_test_top1",
        "target_VI_top1",
        "top3",
        "VI_test_top3",
        "target_VI_top3",
        "top5",
        "VI_test_top5",
        "target_VI_top5",
        "dominant_ratio",
        "normalized_entropy",
        "best_epoch",
    ]:
        if key in m:
            row[key] = m[key]

    rows.append(row)

metrics = pd.DataFrame(rows)
display(metrics.sort_values(["subject", "method"]))
print("rows:", len(metrics))

,subject,method,path,balanced_accuracy,top1,top3,top5,best_epoch,dominant_ratio,normalized_entropy
81,1,gated_residual_vs_to_vi,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...,0.134921,0.134921,0.341270,0.547619,12,0.158730,0.983875
39,1,raw_tf_vi_only,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.103175,0.103175,0.341270,0.515873,19,0.222222,0.947199
40,1,raw_tf_vs_to_vi,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.126984,0.126984,0.349206,0.579365,57,0.230159,0.859863
35,1,raw_vi_only,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.095238,0.095238,0.341270,0.595238,62,0.579365,0.434849
36,1,raw_vs_to_vi,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.111111,0.111111,0.333333,0.555556,1,0.769841,0.245522
...,...,...,...,...,...,...,...,...,...,...
74,29,unknown,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.125000,0.125000,0.361111,0.583333,0,0.194444,0.877476
77,29,unknown,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.111111,0.111111,0.277778,0.583333,4,0.611111,0.459291
78,29,unknown,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,0.097222,0.097222,0.375000,0.569444,0,0.347222,0.598642
19,29,vi_only_other,/content/drive/MyDrive/vsvi_data/vi_encoder_tr...,0.138889,0.138889,0.444444,0.638889,22,NaN,NaN


rows: 87


In [13]:
import numpy as np
import pandas as pd

def first_existing(row, cols):
    for c in cols:
        if c in row and pd.notna(row[c]):
            return row[c]
    return np.nan

norm_rows = []

for _, r in metrics.iterrows():
    norm_rows.append({
        "subject": int(r["subject"]),
        "method": r["method"],
        "BAC": first_existing(r, ["balanced_accuracy", "test_BAC", "VI_test_BAC", "target_VI_BAC"]),
        "Top1": first_existing(r, ["top1", "VI_test_top1", "target_VI_top1"]),
        "Top3": first_existing(r, ["top3", "VI_test_top3", "target_VI_top3"]),
        "Top5": first_existing(r, ["top5", "VI_test_top5", "target_VI_top5"]),
        "dominant_ratio": first_existing(r, ["dominant_ratio"]),
        "entropy": first_existing(r, ["normalized_entropy"]),
        "best_epoch": first_existing(r, ["best_epoch"]),
        "path": r["path"],
    })

enc = pd.DataFrame(norm_rows)
display(enc.sort_values(["subject", "method"]))

,subject,method,BAC,Top1,Top3,Top5,dominant_ratio,entropy,best_epoch,path
81,1,gated_residual_vs_to_vi,0.134921,0.134921,0.341270,0.547619,0.158730,0.983875,12,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
39,1,raw_tf_vi_only,0.103175,0.103175,0.341270,0.515873,0.222222,0.947199,19,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
40,1,raw_tf_vs_to_vi,0.126984,0.126984,0.349206,0.579365,0.230159,0.859863,57,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
35,1,raw_vi_only,0.095238,0.095238,0.341270,0.595238,0.579365,0.434849,62,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
36,1,raw_vs_to_vi,0.111111,0.111111,0.333333,0.555556,0.769841,0.245522,1,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
...,...,...,...,...,...,...,...,...,...,...
74,29,unknown,0.125000,0.125000,0.361111,0.583333,0.194444,0.877476,0,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
77,29,unknown,0.111111,0.111111,0.277778,0.583333,0.611111,0.459291,4,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
78,29,unknown,0.097222,0.097222,0.375000,0.569444,0.347222,0.598642,0,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
19,29,vi_only_other,0.138889,0.138889,0.444444,0.638889,NaN,NaN,22,/content/drive/MyDrive/vsvi_data/vi_encoder_tr...


In [14]:
summary = (
    enc.groupby("method")
    .agg(
        n=("subject", "nunique"),
        BAC_mean=("BAC", "mean"),
        BAC_median=("BAC", "median"),
        BAC_std=("BAC", "std"),
        Top3_mean=("Top3", "mean"),
        Top5_mean=("Top5", "mean"),
        dominant_mean=("dominant_ratio", "mean"),
        entropy_mean=("entropy", "mean"),
    )
    .sort_values("BAC_mean", ascending=False)
)

display(summary)

,n,BAC_mean,BAC_median,BAC_std,Top3_mean,Top5_mean,dominant_mean,entropy_mean
method,,,,,,,,
raw_tf_vs_to_vi,7,0.150586,0.140741,0.072903,0.373847,0.565193,0.287075,0.836703
gated_residual_vs_to_vi,6,0.146164,0.143849,0.034786,0.419643,0.611772,0.227072,0.907678
raw_tf_vi_only,7,0.142177,0.125000,0.039307,0.386130,0.579497,0.314928,0.815278
vi_only_other,7,0.138856,0.138889,0.031438,0.383780,0.602761,0.192593,0.963301
unknown,7,0.137989,0.129630,0.039857,0.372874,0.603456,0.487516,0.585989
vs_to_vi_other,7,0.135384,0.118519,0.054510,0.360202,0.571346,0.207407,0.945264
raw_vi_only,7,0.121523,0.111111,0.039964,0.371977,0.595711,0.566723,0.470386
raw_vs_to_vi,7,0.115363,0.111111,0.012937,0.354214,0.581085,0.599868,0.435658


In [15]:
wide = enc.pivot_table(
    index="subject",
    columns="method",
    values="BAC",
    aggfunc="first",
)

display(wide)

comparisons = pd.DataFrame({
    "subject": wide.index,
    "raw_tf_vs_to_vi_minus_raw_tf_vi_only": (
        wide.get("raw_tf_vs_to_vi") - wide.get("raw_tf_vi_only")
    ),
    "raw_tf_vi_only_minus_raw_vi_only": (
        wide.get("raw_tf_vi_only") - wide.get("raw_vi_only")
    ),
    "raw_tf_vs_to_vi_minus_raw_vs_to_vi": (
        wide.get("raw_tf_vs_to_vi") - wide.get("raw_vs_to_vi")
    ),
    "gated_minus_raw_tf_vi_only": (
        wide.get("gated_residual_vs_to_vi") - wide.get("raw_tf_vi_only")
    ),
    "gated_minus_raw_tf_vs_to_vi": (
        wide.get("gated_residual_vs_to_vi") - wide.get("raw_tf_vs_to_vi")
    ),
})

display(comparisons)

delta_summary = comparisons.drop(columns=["subject"]).agg(["count", "mean", "median", "std"]).T
display(delta_summary)

method,gated_residual_vs_to_vi,raw_tf_vi_only,raw_tf_vs_to_vi,raw_vi_only,raw_vs_to_vi,unknown,vi_only_other,vs_to_vi_other
subject,,,,,,,,
1,0.134921,0.103175,0.126984,0.095238,0.111111,0.111111,0.119048,0.087302
2,0.158730,0.158730,0.142857,0.206349,0.126984,0.142857,0.182540,0.214286
9,0.203704,0.222222,0.296296,0.111111,0.092593,0.166667,0.185185,0.222222
18,0.101852,0.120370,0.055556,0.111111,0.129630,0.138889,0.138889,0.101852
24,NaN,0.140741,0.140741,0.118519,0.111111,0.103704,0.103704,0.125926
28,0.152778,0.125000,0.125000,0.083333,0.111111,0.138889,0.138889,0.111111
29,0.125000,0.125000,0.166667,0.125000,0.125000,0.138889,0.138889,0.138889


,subject,raw_tf_vs_to_vi_minus_raw_tf_vi_only,raw_tf_vi_only_minus_raw_vi_only,raw_tf_vs_to_vi_minus_raw_vs_to_vi,gated_minus_raw_tf_vi_only,gated_minus_raw_tf_vs_to_vi
subject,,,,,,
1,1,0.023810,0.007937,0.015873,0.031746,0.007937
2,2,-0.015873,-0.047619,0.015873,0.000000,0.015873
9,9,0.074074,0.111111,0.203704,-0.018519,-0.092593
18,18,-0.064815,0.009259,-0.074074,-0.018519,0.046296
24,24,0.000000,0.022222,0.029630,NaN,NaN
28,28,0.000000,0.041667,0.013889,0.027778,0.027778
29,29,0.041667,0.000000,0.041667,0.000000,-0.041667


,count,mean,median,std
raw_tf_vs_to_vi_minus_raw_tf_vi_only,7.0,0.008409,0.000000,0.044249
raw_tf_vi_only_minus_raw_vi_only,7.0,0.020654,0.009259,0.048315
raw_tf_vs_to_vi_minus_raw_vs_to_vi,7.0,0.035223,0.015873,0.083275
gated_minus_raw_tf_vi_only,6.0,0.003748,0.000000,0.021822
gated_minus_raw_tf_vs_to_vi,6.0,-0.006063,0.011905,0.051612


In [16]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

enc.to_csv(OUT_DIR / "encoder_subject_level_metrics.csv", index=False)
summary.to_csv(OUT_DIR / "encoder_method_summary.csv")
comparisons.to_csv(OUT_DIR / "encoder_transfer_deltas.csv", index=False)
delta_summary.to_csv(OUT_DIR / "encoder_transfer_delta_summary.csv")

print("saved to:", OUT_DIR)

saved to: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis


In [17]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, binomtest, spearmanr

def paired_report(x, name):
    x = pd.Series(x).dropna()
    n = len(x)
    mean = x.mean()
    median = x.median()
    std = x.std()
    better = int((x > 0).sum())
    tie = int((x == 0).sum())
    worse = int((x < 0).sum())

    # sign test: ties removed
    nonzero = x[x != 0]
    if len(nonzero) > 0:
        sign_p = binomtest(int((nonzero > 0).sum()), len(nonzero), 0.5, alternative="two-sided").pvalue
    else:
        sign_p = np.nan

    # wilcoxon
    try:
        wil_p = wilcoxon(x, zero_method="wilcox", alternative="two-sided").pvalue
    except Exception:
        wil_p = np.nan

    return {
        "comparison": name,
        "n": n,
        "mean": mean,
        "median": median,
        "std": std,
        "better": better,
        "tie": tie,
        "worse": worse,
        "sign_p": sign_p,
        "wilcoxon_p": wil_p,
    }

tests = []

for col in [
    "raw_tf_vs_to_vi_minus_raw_tf_vi_only",
    "raw_tf_vi_only_minus_raw_vi_only",
    "raw_tf_vs_to_vi_minus_raw_vs_to_vi",
    "gated_minus_raw_tf_vi_only",
    "gated_minus_raw_tf_vs_to_vi",
]:
    tests.append(paired_report(comparisons[col], col))

tests = pd.DataFrame(tests)
display(tests)

,comparison,n,mean,median,std,better,tie,worse,sign_p,wilcoxon_p
0,raw_tf_vs_to_vi_minus_raw_tf_vi_only,7,0.008409,0.000000,0.044249,3,2,2,1.00000,0.625000
1,raw_tf_vi_only_minus_raw_vi_only,7,0.020654,0.009259,0.048315,5,1,1,0.21875,0.312500
2,raw_tf_vs_to_vi_minus_raw_vs_to_vi,7,0.035223,0.015873,0.083275,6,0,1,0.12500,0.203125
3,gated_minus_raw_tf_vi_only,6,0.003748,0.000000,0.021822,2,2,2,1.00000,0.625000
4,gated_minus_raw_tf_vs_to_vi,6,-0.006063,0.011905,0.051612,4,0,2,0.68750,1.000000


In [18]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

wide.to_csv(OUT_DIR / "encoder_wide_subject_metrics.csv")
comparisons.to_csv(OUT_DIR / "encoder_transfer_deltas.csv", index=False)
delta_summary.to_csv(OUT_DIR / "encoder_transfer_delta_summary.csv")
tests.to_csv(OUT_DIR / "encoder_transfer_stat_tests.csv", index=False)

print("saved:", OUT_DIR)

saved: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis


In [19]:
paper_rows = [
    {
        "result": "TF benefit for VI-only",
        "comparison": "raw_tf_vi_only - raw_vi_only",
        "mean_delta_BAC": comparisons["raw_tf_vi_only_minus_raw_vi_only"].mean(),
        "median_delta_BAC": comparisons["raw_tf_vi_only_minus_raw_vi_only"].median(),
    },
    {
        "result": "TF benefit for VS→VI",
        "comparison": "raw_tf_vs_to_vi - raw_vs_to_vi",
        "mean_delta_BAC": comparisons["raw_tf_vs_to_vi_minus_raw_vs_to_vi"].mean(),
        "median_delta_BAC": comparisons["raw_tf_vs_to_vi_minus_raw_vs_to_vi"].median(),
    },
    {
        "result": "VS→VI transfer effect under raw_tf",
        "comparison": "raw_tf_vs_to_vi - raw_tf_vi_only",
        "mean_delta_BAC": comparisons["raw_tf_vs_to_vi_minus_raw_tf_vi_only"].mean(),
        "median_delta_BAC": comparisons["raw_tf_vs_to_vi_minus_raw_tf_vi_only"].median(),
    },
    {
        "result": "Gated residual over raw_tf VI-only",
        "comparison": "gated - raw_tf_vi_only",
        "mean_delta_BAC": comparisons["gated_minus_raw_tf_vi_only"].mean(),
        "median_delta_BAC": comparisons["gated_minus_raw_tf_vi_only"].median(),
    },
]

paper = pd.DataFrame(paper_rows)
display(paper)
paper.to_csv(OUT_DIR / "paper_compact_encoder_results.csv", index=False)

,result,comparison,mean_delta_BAC,median_delta_BAC
0,TF benefit for VI-only,raw_tf_vi_only - raw_vi_only,0.020654,0.009259
1,TF benefit for VS→VI,raw_tf_vs_to_vi - raw_vs_to_vi,0.035223,0.015873
2,VS→VI transfer effect under raw_tf,raw_tf_vs_to_vi - raw_tf_vi_only,0.008409,0.000000
3,Gated residual over raw_tf VI-only,gated - raw_tf_vi_only,0.003748,0.000000


In [20]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

md = f"""# Final VS-to-VI Transfer Summary

## Encoder-Level Findings

| Comparison | n | Mean ΔBAC | Median ΔBAC | Better / Tie / Worse | Sign p | Wilcoxon p |
|---|---:|---:|---:|---:|---:|---:|
"""

for _, r in tests.iterrows():
    md += (
        f"| {r['comparison']} | {int(r['n'])} | "
        f"{r['mean']:.4f} | {r['median']:.4f} | "
        f"{int(r['better'])}/{int(r['tie'])}/{int(r['worse'])} | "
        f"{r['sign_p']:.4f} | {r['wilcoxon_p']:.4f} |\n"
    )

md += """

## Interpretation

- Raw+TF representation improved over raw EEG, especially in the VS→VI condition.
- Direct VS→VI transfer over VI-only under raw+TF showed only a small, non-significant mean gain.
- Transfer effects were subject-dependent.
- Gated residual transfer did not yield consistent additional improvement.
- Downstream image generation showed unstable semantic alignment: encoder-level improvements did not reliably translate into class-correct generation.

## Main Claim

Time-frequency augmentation makes VS→VI transfer more viable, but the current transfer effect remains weak and subject-dependent. The main bottleneck is semantic alignment between VI representations and the VS-conditioned generator space.
"""

path = OUT_DIR / "final_vs_to_vi_summary.md"
path.write_text(md, encoding="utf-8")

print("saved:", path)
print(md)

saved: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis/final_vs_to_vi_summary.md
# Final VS-to-VI Transfer Summary

## Encoder-Level Findings

| Comparison | n | Mean ΔBAC | Median ΔBAC | Better / Tie / Worse | Sign p | Wilcoxon p |
|---|---:|---:|---:|---:|---:|---:|
| raw_tf_vs_to_vi_minus_raw_tf_vi_only | 7 | 0.0084 | 0.0000 | 3/2/2 | 1.0000 | 0.6250 |
| raw_tf_vi_only_minus_raw_vi_only | 7 | 0.0207 | 0.0093 | 5/1/1 | 0.2188 | 0.3125 |
| raw_tf_vs_to_vi_minus_raw_vs_to_vi | 7 | 0.0352 | 0.0159 | 6/0/1 | 0.1250 | 0.2031 |
| gated_minus_raw_tf_vi_only | 6 | 0.0037 | 0.0000 | 2/2/2 | 1.0000 | 0.6250 |
| gated_minus_raw_tf_vs_to_vi | 6 | -0.0061 | 0.0119 | 4/0/2 | 0.6875 | 1.0000 |


## Interpretation

- Raw+TF representation improved over raw EEG, especially in the VS→VI condition.
- Direct VS→VI transfer over VI-only under raw+TF showed only a small, non-significant mean gain.
- Transfer effects were subject-dependent.
- Gated residual transfer did not yield consistent additi

Validation-selected safe transfer + raw_tf

In [21]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data")

SUBJECTS = [1, 2, 9, 18, 24, 28, 29]

CANDIDATES = {
    "raw_tf_vi_only": [
        "vi_rawtf_confirmatory_multi/seed42/S{sid:02d}/raw_tf/vi_only/metrics.json",
        "vi_tf_representation_ablation/seed42/S{sid:02d}/raw_tf/vi_only/metrics.json",
    ],
    "raw_tf_vs_to_vi": [
        "vi_rawtf_confirmatory_multi/seed42/S{sid:02d}/raw_tf/vs_to_vi/metrics.json",
        "vi_tf_representation_ablation/seed42/S{sid:02d}/raw_tf/vs_to_vi/metrics.json",
    ],
    "gated_residual": [
        "vi_rawtf_gated_residual_confirmatory/seed42/S{sid:02d}/gated_residual_vs_to_vi/metrics.json",
        "vi_rawtf_gated_residual_s24/seed42/S{sid:02d}/gated_residual_vs_to_vi/metrics.json",
    ],
}

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

def first_value(flat, keys):
    for k in keys:
        if k in flat and flat[k] is not None:
            return flat[k]
    return np.nan

rows = []

for sid in SUBJECTS:
    for method, patterns in CANDIDATES.items():
        path = None
        for pat in patterns:
            p = ROOT / pat.format(sid=sid)
            if p.exists():
                path = p
                break

        if path is None:
            rows.append({
                "subject": sid,
                "method": method,
                "exists": False,
                "path": None,
            })
            continue

        flat = flatten(load_json(path))

        val_bac = first_value(flat, [
            "val_BAC",
            "validation_BAC",
            "validation_token_BAC",
            "best_vi_val_token_bac",
            "best_vi_val_latent_bac",
            "val_balanced_accuracy",
            "val.balanced_accuracy",
        ])

        test_bac = first_value(flat, [
            "test_BAC",
            "VI_test_BAC",
            "balanced_accuracy",
            "target_VI_BAC",
            "test.balanced_accuracy",
        ])

        top3 = first_value(flat, [
            "test_top3",
            "VI_test_top3",
            "top3",
            "target_VI_top3",
        ])

        top5 = first_value(flat, [
            "test_top5",
            "VI_test_top5",
            "top5",
            "target_VI_top5",
        ])

        rows.append({
            "subject": sid,
            "method": method,
            "exists": True,
            "val_BAC": val_bac,
            "test_BAC": test_bac,
            "test_Top3": top3,
            "test_Top5": top5,
            "path": str(path),
        })

safe_raw = pd.DataFrame(rows)
display(safe_raw.sort_values(["subject", "method"]))

,subject,method,exists,val_BAC,test_BAC,test_Top3,test_Top5,path
2,1,gated_residual,True,NaN,0.134921,0.341270,0.547619,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
0,1,raw_tf_vi_only,True,NaN,0.103175,0.341270,0.515873,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
1,1,raw_tf_vs_to_vi,True,NaN,0.126984,0.349206,0.579365,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
5,2,gated_residual,True,NaN,0.158730,0.523810,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
3,2,raw_tf_vi_only,True,NaN,0.158730,0.507937,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
4,2,raw_tf_vs_to_vi,True,NaN,0.142857,0.349206,0.571429,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
8,9,gated_residual,True,NaN,0.203704,0.425926,0.555556,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
6,9,raw_tf_vi_only,True,NaN,0.222222,0.407407,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
7,9,raw_tf_vs_to_vi,True,NaN,0.296296,0.462963,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
11,18,gated_residual,True,NaN,0.101852,0.324074,0.527778,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...


In [22]:
nan_rows = safe_raw[safe_raw["exists"] & safe_raw["val_BAC"].isna()]
display(nan_rows)

for _, r in nan_rows.iterrows():
    print("\n", r["subject"], r["method"], r["path"])
    flat = flatten(load_json(Path(r["path"])))
    for k in sorted(flat.keys()):
        if any(x in k.lower() for x in ["val", "bac", "top", "acc", "best"]):
            print(k, "=", flat[k])

,subject,method,exists,val_BAC,test_BAC,test_Top3,test_Top5,path
0,1,raw_tf_vi_only,True,NaN,0.103175,0.341270,0.515873,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
1,1,raw_tf_vs_to_vi,True,NaN,0.126984,0.349206,0.579365,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
2,1,gated_residual,True,NaN,0.134921,0.341270,0.547619,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
3,2,raw_tf_vi_only,True,NaN,0.158730,0.507937,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
4,2,raw_tf_vs_to_vi,True,NaN,0.142857,0.349206,0.571429,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
5,2,gated_residual,True,NaN,0.158730,0.523810,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
6,9,raw_tf_vi_only,True,NaN,0.222222,0.407407,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
7,9,raw_tf_vs_to_vi,True,NaN,0.296296,0.462963,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
8,9,gated_residual,True,NaN,0.203704,0.425926,0.555556,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
9,18,raw_tf_vi_only,True,NaN,0.120370,0.314815,0.518519,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...



 1 raw_tf_vi_only /content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only/metrics.json
balanced_accuracy = 0.10317460317460317
best_epoch = 19
n_val = 117
nonfinite_removed.val = 0
top1 = 0.10317460317460317
top3 = 0.3412698412698413
top5 = 0.5158730158730159

 1 raw_tf_vs_to_vi /content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vs_to_vi/metrics.json
balanced_accuracy = 0.12698412698412698
best_epoch = 57
n_val = 117
nonfinite_removed.val = 0
top1 = 0.12698412698412698
top3 = 0.3492063492063492
top5 = 0.5793650793650794

 1 gated_residual /content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S01/gated_residual_vs_to_vi/metrics.json
balanced_accuracy = 0.1349206349206349
best_epoch = 12
best_phase = joint
best_validation.balanced_accuracy = 0.17094017094017092
best_validation.dominant_label = 0
best_validation.dominant_ratio = 0.1452991452991453
best_validation.mean_true_margin = -0.963246910379101
best_v

In [23]:
from pathlib import Path
import pandas as pd

for _, r in safe_raw[safe_raw["exists"]].iterrows():
    folder = Path(r["path"]).parent

    print("\n" + "=" * 100)
    print("S", r["subject"], r["method"])
    print(folder)
    print("=" * 100)

    csvs = sorted(folder.glob("*.csv"))
    if not csvs:
        print("no csv")
        continue

    for csv in csvs:
        try:
            df = pd.read_csv(csv)
        except Exception as e:
            print("failed:", csv.name, e)
            continue

        print("\n", csv.name)
        print("columns:", df.columns.tolist())
        display(df.tail(3))


S 1 raw_tf_vi_only
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only

 confusion.csv
columns: ['1', '2', '3', '2.1', '1.1', '1.2', '1.3', '3.1', '0']


,1,2,3,2.1,1.1,1.2,1.3,3.1,0
5,3,0,1,1,3,0,3,3,0
6,6,1,1,3,1,1,1,0,0
7,3,1,0,3,1,0,2,3,1



 history.csv
columns: ['epoch', 'train_loss', 'val_balanced_accuracy', 'val_top1', 'val_top3', 'val_top5', 'val_normalized_entropy', 'val_dominant_ratio']


,epoch,train_loss,val_balanced_accuracy,val_top1,val_top3,val_top5,val_normalized_entropy,val_dominant_ratio
36,37,0.577347,0.119658,0.119658,0.350427,0.529915,0.990316,0.145299
37,38,0.569877,0.145299,0.145299,0.367521,0.555556,0.990915,0.145299
38,39,0.563856,0.136752,0.136752,0.384615,0.572650,0.971293,0.188034



 predictions.csv
columns: ['sample_index', 'true_label', 'pred_label', 'correct', 'true_score', 'true_margin']


...

,3,1,2,2.1,0,0.1,0.2,0.3,0.4
5,1,3,1,2,0,0,0,1,0
6,1,1,1,1,0,0,0,0,4
7,3,1,0,2,0,0,0,0,2



 history.csv
columns: ['epoch', 'phase', 'train_loss', 'train_ce', 'train_supcon', 'gate_value', 'val_balanced_accuracy', 'val_top1', 'val_top3', 'val_top5', 'val_normalized_entropy', 'val_dominant_ratio']


,epoch,phase,train_loss,train_ce,train_supcon,gate_value,val_balanced_accuracy,val_top1,val_top3,val_top5,val_normalized_entropy,val_dominant_ratio
28,28,joint,1.741418,1.659737,0.408387,-0.060231,0.095238,0.095238,0.396825,0.603175,0.974923,0.190476
29,29,joint,1.692336,1.621814,0.352595,-0.058976,0.126984,0.126984,0.428571,0.571429,0.983192,0.142857
30,30,joint,1.640917,1.582915,0.289993,-0.057682,0.111111,0.111111,0.412698,0.603175,0.957729,0.174603



 predictions.csv
columns: ['sample_index', 'true_label', 'pred_label', 'correct', 'true_score', 'true_margin']


,sample_index,true_label,pred_label,correct,true_score,true_margin
69,69,8,0,0,-0.006220,-0.104570
70,70,8,1,0,-0.057406,-0.184912
71,71,8,8,1,0.169854,0.045961


In [26]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

def first_json_value(flat, keys):
    for k in keys:
        if k in flat and flat[k] is not None:
            return flat[k]
    return np.nan

def find_val_from_csv(folder):
    candidates = []

    for csv in sorted(folder.glob("*.csv")):
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue

        cols = df.columns.tolist()
        lower = {c.lower(): c for c in cols}

        val_cols = [
            c for c in cols
            if "val" in c.lower()
            and any(x in c.lower() for x in ["bac", "bal", "acc", "top1"])
        ]

        # common names first
        priority = [
            "val_BAC",
            "val_bac",
            "val_balanced_accuracy",
            "validation_BAC",
            "validation_balanced_accuracy",
            "val_top1",
        ]

        ordered = []
        for p in priority:
            if p in cols:
                ordered.append(p)
        for c in val_cols:
            if c not in ordered:
                ordered.append(c)

        for c in ordered:
            s = pd.to_numeric(df[c], errors="coerce").dropna()
            if len(s):
                candidates.append({
                    "csv": str(csv),
                    "column": c,
                    "value_last": float(s.iloc[-1]),
                    "value_max": float(s.max()),
                })

    if not candidates:
        return np.nan, None, None

    # best checkpoint selection에 맞추려면 보통 max validation을 사용
    best = max(candidates, key=lambda x: x["value_max"])
    return best["value_max"], best["csv"], best["column"]

rows = []

for _, r in safe_raw[safe_raw["exists"]].iterrows():
    path = Path(r["path"])
    folder = path.parent
    flat = flatten(load_json(path))

    # gated는 json 안에 best_validation이 있음
    val_bac = first_json_value(flat, [
        "best_validation.balanced_accuracy",
        "best_validation.top1",
        "validation.balanced_accuracy",
        "val_BAC",
        "validation_BAC",
        "val_balanced_accuracy",
    ])
    val_source = "json"
    val_column = None

    # raw 계열은 csv에서 찾기
    if pd.isna(val_bac):
        val_bac, val_source, val_column = find_val_from_csv(folder)

    test_bac = first_json_value(flat, [
        "balanced_accuracy",
        "top1",
        "test_BAC",
        "VI_test_BAC",
        "target_VI_BAC",
    ])
    test_top3 = first_json_value(flat, ["top3", "test_top3", "VI_test_top3"])
    test_top5 = first_json_value(flat, ["top5", "test_top5", "VI_test_top5"])

    rows.append({
        "subject": int(r["subject"]),
        "method": r["method"],
        "val_BAC": val_bac,
        "val_source": val_source,
        "val_column": val_column,
        "test_BAC": test_bac,
        "test_Top3": test_top3,
        "test_Top5": test_top5,
        "path": str(path),
    })

safe_candidates = pd.DataFrame(rows)
display(safe_candidates.sort_values(["subject", "method"]))

,subject,method,val_BAC,val_source,val_column,test_BAC,test_Top3,test_Top5,path
2,1,gated_residual,0.170940,json,None,0.134921,0.341270,0.547619,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
0,1,raw_tf_vi_only,0.162393,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.103175,0.341270,0.515873,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
1,1,raw_tf_vs_to_vi,0.222222,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.126984,0.349206,0.579365,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
5,2,gated_residual,0.247863,json,None,0.158730,0.523810,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
3,2,raw_tf_vi_only,0.247863,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.158730,0.507937,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
4,2,raw_tf_vs_to_vi,0.196581,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.142857,0.349206,0.571429,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
8,9,gated_residual,0.222222,json,None,0.203704,0.425926,0.555556,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...
6,9,raw_tf_vi_only,0.222222,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.222222,0.407407,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
7,9,raw_tf_vs_to_vi,0.185185,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy,0.296296,0.462963,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...
11,18,gated_residual,0.157407,json,None,0.101852,0.324074,0.527778,/content/drive/MyDrive/vsvi_data/vi_rawtf_gate...


In [27]:
missing_val = safe_candidates[safe_candidates["val_BAC"].isna()]
display(missing_val[["subject", "method", "path"]])

assert len(missing_val) == 0, "validation BAC가 없는 후보가 있습니다. 위 CSV column 출력을 확인해야 합니다."

,subject,method,path


In [28]:
select_pool = safe_candidates.dropna(subset=["val_BAC", "test_BAC"]).copy()

idx = select_pool.groupby("subject")["val_BAC"].idxmax()
selected = select_pool.loc[idx].sort_values("subject").reset_index(drop=True)

display(selected[[
    "subject", "method", "val_BAC", "test_BAC", "test_Top3", "test_Top5", "val_source", "val_column"
]])

,subject,method,val_BAC,test_BAC,test_Top3,test_Top5,val_source,val_column
0,1,raw_tf_vs_to_vi,0.222222,0.126984,0.349206,0.579365,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy
1,2,raw_tf_vi_only,0.247863,0.158730,0.507937,0.706349,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy
2,9,raw_tf_vi_only,0.222222,0.222222,0.407407,0.574074,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_balanced_accuracy
3,18,gated_residual,0.157407,0.101852,0.324074,0.527778,json,None
4,24,raw_tf_vs_to_vi,0.200000,0.140741,0.362963,0.518519,/content/drive/MyDrive/vsvi_data/vi_tf_represe...,val_balanced_accuracy
5,28,raw_tf_vs_to_vi,0.174603,0.125000,0.388889,0.555556,/content/drive/MyDrive/vsvi_data/vi_rawtf_conf...,val_top1
6,29,gated_residual,0.222222,0.125000,0.402778,0.625000,json,None


In [29]:
baseline = safe_candidates[safe_candidates["method"] == "raw_tf_vi_only"][[
    "subject", "test_BAC", "test_Top3", "test_Top5"
]].rename(columns={
    "test_BAC": "baseline_test_BAC",
    "test_Top3": "baseline_Top3",
    "test_Top5": "baseline_Top5",
})

comp = selected.merge(baseline, on="subject", how="left")

comp["safe_minus_baseline_BAC"] = comp["test_BAC"] - comp["baseline_test_BAC"]
comp["safe_minus_baseline_Top3"] = comp["test_Top3"] - comp["baseline_Top3"]
comp["safe_minus_baseline_Top5"] = comp["test_Top5"] - comp["baseline_Top5"]

display(comp[[
    "subject", "method", "val_BAC",
    "test_BAC", "baseline_test_BAC", "safe_minus_baseline_BAC",
    "test_Top3", "baseline_Top3", "safe_minus_baseline_Top3",
    "test_Top5", "baseline_Top5", "safe_minus_baseline_Top5",
]])

,subject,method,val_BAC,test_BAC,baseline_test_BAC,safe_minus_baseline_BAC,test_Top3,baseline_Top3,safe_minus_baseline_Top3,test_Top5,baseline_Top5,safe_minus_baseline_Top5
0,1,raw_tf_vs_to_vi,0.222222,0.126984,0.103175,0.023810,0.349206,0.341270,0.007937,0.579365,0.515873,0.063492
1,2,raw_tf_vi_only,0.247863,0.158730,0.158730,0.000000,0.507937,0.507937,0.000000,0.706349,0.706349,0.000000
2,9,raw_tf_vi_only,0.222222,0.222222,0.222222,0.000000,0.407407,0.407407,0.000000,0.574074,0.574074,0.000000
3,18,gated_residual,0.157407,0.101852,0.120370,-0.018519,0.324074,0.314815,0.009259,0.527778,0.518519,0.009259
4,24,raw_tf_vs_to_vi,0.200000,0.140741,0.140741,0.000000,0.362963,0.325926,0.037037,0.518519,0.533333,-0.014815
5,28,raw_tf_vs_to_vi,0.174603,0.125000,0.125000,0.000000,0.388889,0.402778,-0.013889,0.555556,0.569444,-0.013889
6,29,gated_residual,0.222222,0.125000,0.125000,0.000000,0.402778,0.402778,0.000000,0.625000,0.638889,-0.013889


In [30]:
from scipy.stats import binomtest, wilcoxon

def summarize_delta(x, name):
    x = pd.Series(x).dropna()
    nonzero = x[x != 0]

    sign_p = (
        binomtest(int((nonzero > 0).sum()), len(nonzero), 0.5).pvalue
        if len(nonzero) else np.nan
    )

    try:
        wil_p = wilcoxon(x, zero_method="wilcox").pvalue
    except Exception:
        wil_p = np.nan

    return {
        "comparison": name,
        "n": len(x),
        "mean": x.mean(),
        "median": x.median(),
        "std": x.std(),
        "better": int((x > 0).sum()),
        "tie": int((x == 0).sum()),
        "worse": int((x < 0).sum()),
        "sign_p": sign_p,
        "wilcoxon_p": wil_p,
    }

safe_summary = pd.DataFrame([
    summarize_delta(comp["safe_minus_baseline_BAC"], "safe_selected_minus_raw_tf_vi_only_BAC"),
    summarize_delta(comp["safe_minus_baseline_Top3"], "safe_selected_minus_raw_tf_vi_only_Top3"),
    summarize_delta(comp["safe_minus_baseline_Top5"], "safe_selected_minus_raw_tf_vi_only_Top5"),
])

display(safe_summary)
display(selected["method"].value_counts())

,comparison,n,mean,median,std,better,tie,worse,sign_p,wilcoxon_p
0,safe_selected_minus_raw_tf_vi_only_BAC,7,0.000756,0.0,0.012287,1,5,1,1.000,1.000
1,safe_selected_minus_raw_tf_vi_only_Top3,7,0.005763,0.0,0.015710,3,3,1,0.625,0.625
2,safe_selected_minus_raw_tf_vi_only_Top5,7,0.004308,0.0,0.027666,2,2,3,1.000,0.750


,count
method,
raw_tf_vs_to_vi,3
raw_tf_vi_only,2
gated_residual,2


In [31]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")

safe_candidates.to_csv(OUT_DIR / "safe_transfer_candidate_metrics.csv", index=False)
selected.to_csv(OUT_DIR / "safe_transfer_selected_by_validation.csv", index=False)
comp.to_csv(OUT_DIR / "safe_transfer_vs_baseline.csv", index=False)
safe_summary.to_csv(OUT_DIR / "safe_transfer_summary.csv", index=False)

print("saved:", OUT_DIR)

saved: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis


In [32]:
path = OUT_DIR / "final_vs_to_vi_summary.md"
md = path.read_text(encoding="utf-8")

addition = """

## Validation-Selected Safe Transfer

Validation-selected model choice did not materially improve over the raw+TF VI-only baseline.

| Metric | Mean Δ | Median Δ | Better / Tie / Worse | Sign p | Wilcoxon p |
|---|---:|---:|---:|---:|---:|
"""

for _, r in safe_summary.iterrows():
    addition += (
        f"| {r['comparison']} | {r['mean']:.4f} | {r['median']:.4f} | "
        f"{int(r['better'])}/{int(r['tie'])}/{int(r['worse'])} | "
        f"{r['sign_p']:.4f} | {r['wilcoxon_p']:.4f} |\n"
    )

addition += """

Model selection counts:
- raw_tf_vs_to_vi: 3
- raw_tf_vi_only: 2
- gated_residual: 2

Interpretation: validation sometimes selected transfer-based models, but this did not translate into reliable test improvement. This suggests that the validation signal is insufficient for identifying subject-specific transfer benefit under the current setup.
"""

path.write_text(md + addition, encoding="utf-8")
print("updated:", path)

updated: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis/final_vs_to_vi_summary.md


In [33]:
from pathlib import Path
import subprocess, sys

PROJECT = Path("/content/vsvi_project")

for p in sorted(PROJECT.glob("*rawtf*.py")):
    print(p)

for p in sorted(PROJECT.glob("*representation*ablation*.py")):
    print(p)

for p in sorted(PROJECT.glob("*transfer*.py")):
    print(p)

/content/vsvi_project/eval_vi_rawtf_downstream_generation.py
/content/vsvi_project/run_vi_rawtf_ccmmd.py
/content/vsvi_project/run_vi_rawtf_gated_residual.py
/content/vsvi_project/run_vi_rawtf_safe_transfer.py
/content/vsvi_project/summarize_vi_rawtf_ccmmd.py
/content/vsvi_project/summarize_vi_rawtf_downstream_generation.py
/content/vsvi_project/summarize_vi_rawtf_gated_residual.py
/content/vsvi_project/summarize_vi_rawtf_safe_transfer.py
/content/vsvi_project/train_vi_rawtf_ccmmd.py
/content/vsvi_project/train_vi_rawtf_gated_residual.py
/content/vsvi_project/train_vi_rawtf_safe_transfer.py
/content/vsvi_project/run_vi_tf_representation_ablation.py
/content/vsvi_project/summarize_vi_tf_representation_ablation.py
/content/vsvi_project/train_vi_tf_representation_ablation.py
/content/vsvi_project/eval_vs_re_exp25_vi_transfer.py
/content/vsvi_project/run_multi_session_vi_encoder_transfer.py
/content/vsvi_project/run_vi_rawtf_safe_transfer.py
/content/vsvi_project/summarize_vi_encoder_trans

In [34]:
import subprocess, sys

SCRIPT = "/content/vsvi_project/train_vi_tf_representation_ablation.py"

result = subprocess.run(
    [sys.executable, "-u", SCRIPT, "--help"],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

RETURN CODE: 0
usage: train_vi_tf_representation_ablation.py [-h] --stage
                                              {vs_pretrain,zero_shot,vi_only,vs_to_vi}
                                              --representation {raw,tf,raw_tf}
                                              --subject_id SUBJECT_ID
                                              --vs_root VS_ROOT --vi_root
                                              VI_ROOT --out_dir OUT_DIR
                                              [--init_ckpt INIT_CKPT]
                                              [--seed SEED]
                                              [--sampling_rate SAMPLING_RATE]
                                              [--n_ch N_CH] [--epochs EPOCHS]
                                              [--batch_size BATCH_SIZE]
                                              [--lr LR]
                                              [--weight_decay WEIGHT_DECAY]
                                              [--patie

In [37]:
import subprocess, sys
from pathlib import Path

SCRIPT = "/content/vsvi_project/train_vi_tf_representation_ablation.py"

cmd = [
    sys.executable, "-u", SCRIPT,
    "--stage", "vi_only",
    "--representation", "raw_tf",
    "--subject_id", "9",
    "--vs_root", "/content/drive/MyDrive/vsvi_data/preproc_vs_re",
    "--vi_root", "/content/drive/MyDrive/vsvi_data/preproc_vi_re",
    "--out_dir", "/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed7/S09/raw_tf/vi_only",
    "--seed", "7",
    "--epochs", "80",
    "--batch_size", "32",
    "--patience", "15",
    "--fp16",
]

result = subprocess.run(cmd, text=True, capture_output=True)

print("RETURN CODE:", result.returncode)
print("STDOUT LAST 5000\n", result.stdout[-5000:])
print("STDERR LAST 5000\n", result.stderr[-5000:])

RETURN CODE: 0
STDOUT LAST 5000
   [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(135, 32, 2048)
  [dataset] S09  trials=540  eff_sessions=4
  [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(135, 32, 2048)
  [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(135, 32, 2048)
[INFO] stage=vi_only representation=raw_tf subject=S09 domain=vi sessions=4 device=cuda
[INFO] split train=432 val=54 test=54
  epoch=001 loss=2.6856 val_BAC=0.0741 val@3=0.3519 dom=0.574
  epoch=010 loss=2.1930 val_BAC=0.1852 val@3=0.4259 dom=0.296
  epoch=020 loss=1.4229 val_BAC=0.2222 val@3=0.5370 dom=0.204
  epoch=030

In [38]:
import subprocess, sys
from pathlib import Path

SCRIPT = "/content/vsvi_project/train_vi_tf_representation_ablation.py"

SUBJECTS = [1, 2, 9, 18, 24, 28, 29]
SEEDS = [7, 42, 123]

VS_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vs_re"
VI_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
ROOT_OUT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep")

for seed in SEEDS:
    for sid in SUBJECTS:
        out_dir = ROOT_OUT / f"seed{seed}" / f"S{sid:02d}" / "raw_tf" / "vi_only"
        metrics = out_dir / "metrics.json"

        if metrics.exists():
            print(f"[SKIP] S{sid:02d} seed={seed}")
            continue

        print("\n" + "=" * 100)
        print(f"START S{sid:02d} seed={seed} raw_tf vi_only")
        print("=" * 100)

        cmd = [
            sys.executable, "-u", SCRIPT,
            "--stage", "vi_only",
            "--representation", "raw_tf",
            "--subject_id", str(sid),
            "--vs_root", VS_ROOT,
            "--vi_root", VI_ROOT,
            "--out_dir", str(out_dir),
            "--seed", str(seed),
            "--epochs", "80",
            "--batch_size", "32",
            "--patience", "15",
            "--fp16",
        ]

        result = subprocess.run(cmd)
        print("RETURN CODE:", result.returncode)
        assert result.returncode == 0


START S01 seed=7 raw_tf vi_only
RETURN CODE: 0

START S02 seed=7 raw_tf vi_only
RETURN CODE: 0
[SKIP] S09 seed=7

START S18 seed=7 raw_tf vi_only
RETURN CODE: 0

START S24 seed=7 raw_tf vi_only
RETURN CODE: 0

START S28 seed=7 raw_tf vi_only
RETURN CODE: 0

START S29 seed=7 raw_tf vi_only
RETURN CODE: 0

START S01 seed=42 raw_tf vi_only
RETURN CODE: 0

START S02 seed=42 raw_tf vi_only
RETURN CODE: 0

START S09 seed=42 raw_tf vi_only
RETURN CODE: 0

START S18 seed=42 raw_tf vi_only
RETURN CODE: 0

START S24 seed=42 raw_tf vi_only
RETURN CODE: 0

START S28 seed=42 raw_tf vi_only
RETURN CODE: 0

START S29 seed=42 raw_tf vi_only
RETURN CODE: 0

START S01 seed=123 raw_tf vi_only
RETURN CODE: 0

START S02 seed=123 raw_tf vi_only
RETURN CODE: 0

START S09 seed=123 raw_tf vi_only
RETURN CODE: 0

START S18 seed=123 raw_tf vi_only
RETURN CODE: 0

START S24 seed=123 raw_tf vi_only
RETURN CODE: 0

START S28 seed=123 raw_tf vi_only
RETURN CODE: 0

START S29 seed=123 raw_tf vi_only
RETURN CODE: 0


In [39]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep")

metrics = sorted(ROOT.rglob("metrics.json"))
print("metrics:", len(metrics), "/ expected", 21)

for p in metrics:
    print(p)

metrics: 21 / expected 21
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S01/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S02/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S09/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S18/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S24/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S28/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed123/S29/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed42/S01/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed42/S02/raw_tf/vi_only/metrics.json
/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep/seed42

In [40]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_only_seed_sweep")

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

def first(flat, keys):
    for k in keys:
        if k in flat and flat[k] is not None:
            return flat[k]
    return np.nan

rows = []

for p in sorted(ROOT.rglob("metrics.json")):
    parts = p.parts

    subj = next((x for x in parts if x.startswith("S") and x[1:].isdigit()), None)
    seed_part = next((x for x in parts if x.startswith("seed")), None)
    seed = int(seed_part.replace("seed", "")) if seed_part else None

    with open(p, "r", encoding="utf-8") as f:
        flat = flatten(json.load(f))

    rows.append({
        "subject": int(subj[1:]),
        "seed": seed,
        "BAC": first(flat, ["balanced_accuracy", "test_BAC", "VI_test_BAC", "top1"]),
        "Top3": first(flat, ["top3", "test_top3", "VI_test_top3"]),
        "Top5": first(flat, ["top5", "test_top5", "VI_test_top5"]),
        "best_epoch": first(flat, ["best_epoch"]),
        "path": str(p),
    })

sweep = pd.DataFrame(rows).sort_values(["subject", "seed"])
display(sweep)

,subject,seed,BAC,Top3,Top5,best_epoch,path
14,1,7,0.158730,0.349206,0.523810,25,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
7,1,42,0.119048,0.349206,0.547619,19,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
0,1,123,0.119048,0.333333,0.539683,13,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
15,2,7,0.206349,0.476190,0.698413,7,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
8,2,42,0.182540,0.452381,0.690476,11,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
1,2,123,0.166667,0.523810,0.698413,9,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
16,9,7,0.185185,0.333333,0.518519,20,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
9,9,42,0.222222,0.425926,0.574074,10,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
2,9,123,0.203704,0.333333,0.648148,10,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
17,18,7,0.120370,0.324074,0.574074,1,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...


In [41]:
seed_summary = sweep.groupby("seed").agg(
    n=("subject", "nunique"),
    BAC_mean=("BAC", "mean"),
    BAC_median=("BAC", "median"),
    BAC_std=("BAC", "std"),
    Top3_mean=("Top3", "mean"),
    Top5_mean=("Top5", "mean"),
).sort_values("BAC_mean", ascending=False)

display(seed_summary)

,n,BAC_mean,BAC_median,BAC_std,Top3_mean,Top5_mean
seed,,,,,,
7,7,0.155064,0.158730,0.042473,0.360110,0.579630
42,7,0.144407,0.125000,0.041545,0.370465,0.577929
123,7,0.142800,0.138889,0.037297,0.368481,0.599887


In [42]:
idx = sweep.groupby("subject")["BAC"].idxmax()
best = sweep.loc[idx].sort_values("subject").reset_index(drop=True)

display(best)

best_summary = pd.DataFrame([{
    "n_subjects": best["subject"].nunique(),
    "best_BAC_mean": best["BAC"].mean(),
    "best_BAC_median": best["BAC"].median(),
    "best_Top3_mean": best["Top3"].mean(),
    "best_Top5_mean": best["Top5"].mean(),
}])

display(best_summary)

,subject,seed,BAC,Top3,Top5,best_epoch,path
0,1,7,0.158730,0.349206,0.523810,25,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
1,2,7,0.206349,0.476190,0.698413,7,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
2,9,42,0.222222,0.425926,0.574074,10,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
3,18,7,0.120370,0.324074,0.574074,1,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
4,24,123,0.162963,0.370370,0.600000,11,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
5,28,7,0.180556,0.333333,0.555556,9,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
6,29,7,0.152778,0.430556,0.638889,8,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...


,n_subjects,best_BAC_mean,best_BAC_median,best_Top3_mean,best_Top5_mean
0,7,0.171995,0.162963,0.387094,0.594974


In [43]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis/encoder_wide_subject_metrics.csv")

wide_old = pd.read_csv(BASE_PATH)

baseline = wide_old[["subject", "raw_tf_vi_only"]].rename(columns={
    "raw_tf_vi_only": "old_seed42_BAC"
})

comp_seed = best.merge(baseline, on="subject", how="left")
comp_seed["best_minus_old_seed42_BAC"] = comp_seed["BAC"] - comp_seed["old_seed42_BAC"]

display(comp_seed[[
    "subject", "seed", "BAC", "old_seed42_BAC", "best_minus_old_seed42_BAC",
    "Top3", "Top5", "best_epoch"
]])

delta_seed = comp_seed["best_minus_old_seed42_BAC"]

seed_delta_summary = pd.DataFrame([{
    "comparison": "best_of_3_seeds_minus_old_seed42_raw_tf_vi_only",
    "n": len(delta_seed),
    "mean": delta_seed.mean(),
    "median": delta_seed.median(),
    "std": delta_seed.std(),
    "better": int((delta_seed > 0).sum()),
    "tie": int((delta_seed == 0).sum()),
    "worse": int((delta_seed < 0).sum()),
}])

display(seed_delta_summary)

,subject,seed,BAC,old_seed42_BAC,best_minus_old_seed42_BAC,Top3,Top5,best_epoch
0,1,7,0.158730,0.103175,5.555556e-02,0.349206,0.523810,25
1,2,7,0.206349,0.158730,4.761905e-02,0.476190,0.698413,7
2,9,42,0.222222,0.222222,0.000000e+00,0.425926,0.574074,10
3,18,7,0.120370,0.120370,6.938894e-17,0.324074,0.574074,1
4,24,123,0.162963,0.140741,2.222222e-02,0.370370,0.600000,11
5,28,7,0.180556,0.125000,5.555556e-02,0.333333,0.555556,9
6,29,7,0.152778,0.125000,2.777778e-02,0.430556,0.638889,8


,comparison,n,mean,median,std,better,tie,worse
0,best_of_3_seeds_minus_old_seed42_raw_tf_vi_only,7,0.029819,0.027778,0.024089,6,1,0


In [44]:
idx = sweep.groupby("subject")["BAC"].idxmax()
best = sweep.loc[idx].sort_values("subject").reset_index(drop=True)

seed_summary = sweep.groupby("seed").agg(
    n=("subject", "nunique"),
    BAC_mean=("BAC", "mean"),
    BAC_median=("BAC", "median"),
    BAC_std=("BAC", "std"),
    Top3_mean=("Top3", "mean"),
    Top5_mean=("Top5", "mean"),
).sort_values("BAC_mean", ascending=False)

display(seed_summary)
display(best)

print("best mean BAC:", best["BAC"].mean())
print("best median BAC:", best["BAC"].median())
print("best mean Top3:", best["Top3"].mean())
print("best mean Top5:", best["Top5"].mean())
print("best seed counts:")
print(best["seed"].value_counts())

,n,BAC_mean,BAC_median,BAC_std,Top3_mean,Top5_mean
seed,,,,,,
7,7,0.155064,0.158730,0.042473,0.360110,0.579630
42,7,0.144407,0.125000,0.041545,0.370465,0.577929
123,7,0.142800,0.138889,0.037297,0.368481,0.599887


,subject,seed,BAC,Top3,Top5,best_epoch,path
0,1,7,0.158730,0.349206,0.523810,25,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
1,2,7,0.206349,0.476190,0.698413,7,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
2,9,42,0.222222,0.425926,0.574074,10,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
3,18,7,0.120370,0.324074,0.574074,1,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
4,24,123,0.162963,0.370370,0.600000,11,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
5,28,7,0.180556,0.333333,0.555556,9,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...
6,29,7,0.152778,0.430556,0.638889,8,/content/drive/MyDrive/vsvi_data/vi_rawtf_vi_o...


best mean BAC: 0.17199546485260767
best median BAC: 0.16296296296296295
best mean Top3: 0.3870937263794407
best mean Top5: 0.5949735449735449
best seed counts:
seed
7      5
42     1
123    1
Name: count, dtype: int64


In [45]:
import pandas as pd
from pathlib import Path

old = pd.read_csv("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis/encoder_wide_subject_metrics.csv")

baseline = old[["subject", "raw_tf_vi_only"]].rename(columns={
    "raw_tf_vi_only": "old_seed42_BAC"
})

comp_seed = best.merge(baseline, on="subject", how="left")
comp_seed["best_minus_old_seed42_BAC"] = comp_seed["BAC"] - comp_seed["old_seed42_BAC"]

display(comp_seed[[
    "subject", "seed", "BAC", "old_seed42_BAC", "best_minus_old_seed42_BAC",
    "Top3", "Top5", "best_epoch"
]])

display(comp_seed["best_minus_old_seed42_BAC"].agg(["count", "mean", "median", "std"]))
print("better/tie/worse:",
      (comp_seed["best_minus_old_seed42_BAC"] > 0).sum(),
      (comp_seed["best_minus_old_seed42_BAC"] == 0).sum(),
      (comp_seed["best_minus_old_seed42_BAC"] < 0).sum())

,subject,seed,BAC,old_seed42_BAC,best_minus_old_seed42_BAC,Top3,Top5,best_epoch
0,1,7,0.158730,0.103175,5.555556e-02,0.349206,0.523810,25
1,2,7,0.206349,0.158730,4.761905e-02,0.476190,0.698413,7
2,9,42,0.222222,0.222222,0.000000e+00,0.425926,0.574074,10
3,18,7,0.120370,0.120370,6.938894e-17,0.324074,0.574074,1
4,24,123,0.162963,0.140741,2.222222e-02,0.370370,0.600000,11
5,28,7,0.180556,0.125000,5.555556e-02,0.333333,0.555556,9
6,29,7,0.152778,0.125000,2.777778e-02,0.430556,0.638889,8


,best_minus_old_seed42_BAC
count,7.000000
mean,0.029819
median,0.027778
std,0.024089


better/tie/worse: 6 1 0


In [46]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sweep.to_csv(OUT_DIR / "raw_tf_vi_only_seed_sweep_all.csv", index=False)
seed_summary.to_csv(OUT_DIR / "raw_tf_vi_only_seed_sweep_by_seed.csv")
best.to_csv(OUT_DIR / "raw_tf_vi_only_seed_sweep_best_by_subject.csv", index=False)
comp_seed.to_csv(OUT_DIR / "raw_tf_vi_only_seed_sweep_vs_old_seed42.csv", index=False)

print("saved:", OUT_DIR)

saved: /content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis


In [ ]:
import subprocess, sys
from pathlib import Path

SCRIPT = "/content/vsvi_project/train_vi_tf_representation_ablation.py"

SUBJECTS = [9, 18, 24]
SEED = 7

VS_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vs_re"
VI_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_arch_tuning_pilot")

CONFIGS = {
    "baseline_h256_l128_d03_L2": {
        "hidden_dim": 256,
        "latent_dim": 128,
        "dropout": 0.3,
        "n_layers": 2,
    },
    "wide_h384_l256_d03_L2": {
        "hidden_dim": 384,
        "latent_dim": 256,
        "dropout": 0.3,
        "n_layers": 2,
    },
    "lowdrop_h256_l128_d01_L2": {
        "hidden_dim": 256,
        "latent_dim": 128,
        "dropout": 0.1,
        "n_layers": 2,
    },
    "deep_h256_l128_d03_L3": {
        "hidden_dim": 256,
        "latent_dim": 128,
        "dropout": 0.3,
        "n_layers": 3,
    },
    "wide_lowdrop_h384_l256_d01_L2": {
        "hidden_dim": 384,
        "latent_dim": 256,
        "dropout": 0.1,
        "n_layers": 2,
    },
}

for sid in SUBJECTS:
    for cfg_name, cfg in CONFIGS.items():
        out_dir = OUT_ROOT / f"seed{SEED}" / f"S{sid:02d}" / cfg_name / "raw_tf" / "vi_only"
        metrics = out_dir / "metrics.json"

        if metrics.exists():
            print(f"[SKIP] S{sid:02d} {cfg_name}")
            continue

        print("\n" + "=" * 100)
        print(f"START S{sid:02d} {cfg_name}")
        print("=" * 100)

        cmd = [
            sys.executable, "-u", SCRIPT,
            "--stage", "vi_only",
            "--representation", "raw_tf",
            "--subject_id", str(sid),
            "--vs_root", VS_ROOT,
            "--vi_root", VI_ROOT,
            "--out_dir", str(out_dir),
            "--seed", str(SEED),
            "--epochs", "80",
            "--batch_size", "32",
            "--patience", "15",
            "--hidden_dim", str(cfg["hidden_dim"]),
            "--latent_dim", str(cfg["latent_dim"]),
            "--n_layers", str(cfg["n_layers"]),
            "--dropout", str(cfg["dropout"]),
            "--fp16",
        ]

        result = subprocess.run(cmd)
        print("RETURN CODE:", result.returncode)
        assert result.returncode == 0


START S09 baseline_h256_l128_d03_L2
RETURN CODE: 0

START S09 wide_h384_l256_d03_L2


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_arch_tuning_pilot")
metrics = sorted(ROOT.rglob("metrics.json"))

print("metrics:", len(metrics), "/ expected", 15)
for p in metrics:
    print(p)

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_arch_tuning_pilot")

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

def first(flat, keys):
    for k in keys:
        if k in flat and flat[k] is not None:
            return flat[k]
    return np.nan

rows = []

for p in sorted(ROOT.rglob("metrics.json")):
    parts = p.parts

    subj = next(x for x in parts if x.startswith("S") and x[1:].isdigit())
    cfg_name = parts[parts.index(subj) + 1]
    seed_part = next(x for x in parts if x.startswith("seed"))
    seed = int(seed_part.replace("seed", ""))

    with open(p, "r", encoding="utf-8") as f:
        flat = flatten(json.load(f))

    rows.append({
        "subject": int(subj[1:]),
        "seed": seed,
        "config": cfg_name,
        "BAC": first(flat, ["balanced_accuracy", "top1"]),
        "Top3": first(flat, ["top3"]),
        "Top5": first(flat, ["top5"]),
        "best_epoch": first(flat, ["best_epoch"]),
        "path": str(p),
    })

arch = pd.DataFrame(rows).sort_values(["subject", "config"])
display(arch)

In [ ]:
arch_summary = arch.groupby("config").agg(
    n=("subject", "nunique"),
    BAC_mean=("BAC", "mean"),
    BAC_median=("BAC", "median"),
    BAC_std=("BAC", "std"),
    Top3_mean=("Top3", "mean"),
    Top5_mean=("Top5", "mean"),
).sort_values("BAC_mean", ascending=False)

display(arch_summary)

In [ ]:
idx = arch.groupby("subject")["BAC"].idxmax()
arch_best = arch.loc[idx].sort_values("subject").reset_index(drop=True)

display(arch_best)
display(arch_best["config"].value_counts())

In [ ]:
seed7_baseline = sweep[(sweep["seed"] == 7) & (sweep["subject"].isin([9, 18, 24]))][[
    "subject", "BAC", "Top3", "Top5"
]].rename(columns={
    "BAC": "seed7_default_BAC",
    "Top3": "seed7_default_Top3",
    "Top5": "seed7_default_Top5",
})

comp_arch = arch_best.merge(seed7_baseline, on="subject", how="left")

comp_arch["best_arch_minus_default_BAC"] = comp_arch["BAC"] - comp_arch["seed7_default_BAC"]
comp_arch["best_arch_minus_default_Top3"] = comp_arch["Top3"] - comp_arch["seed7_default_Top3"]
comp_arch["best_arch_minus_default_Top5"] = comp_arch["Top5"] - comp_arch["seed7_default_Top5"]

display(comp_arch[[
    "subject", "config", "BAC", "seed7_default_BAC", "best_arch_minus_default_BAC",
    "Top3", "seed7_default_Top3", "best_arch_minus_default_Top3",
    "Top5", "seed7_default_Top5", "best_arch_minus_default_Top5",
]])

display(comp_arch[[
    "best_arch_minus_default_BAC",
    "best_arch_minus_default_Top3",
    "best_arch_minus_default_Top5",
]].agg(["count", "mean", "median", "std"]))

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

arch.to_csv(OUT_DIR / "raw_tf_arch_tuning_pilot_all.csv", index=False)
arch_summary.to_csv(OUT_DIR / "raw_tf_arch_tuning_pilot_summary.csv")
arch_best.to_csv(OUT_DIR / "raw_tf_arch_tuning_pilot_best_by_subject.csv", index=False)
comp_arch.to_csv(OUT_DIR / "raw_tf_arch_tuning_pilot_vs_default_seed7.csv", index=False)

print("saved:", OUT_DIR)